In [ ]:
# ============================================================
# Hybrid LSTM-GARCH — Step 4: LSTM Model
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import warnings
warnings.filterwarnings('ignore')

# Set seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow version: {tf.__version__}")

In [ ]:
# ── Load GARCH outputs ────────────────────────────────────────
garch_df = pd.read_csv('data/garch_outputs.csv',
                       index_col='Date', parse_dates=True)
print(f"Loaded GARCH outputs: {len(garch_df)} rows")
print(garch_df.head(3))

# ── Build feature matrix ──────────────────────────────────────
# Features fed to LSTM:
# 1. Log return (raw signal)
# 2. GARCH conditional volatility (what GARCH knows)
# 3. Standardised residual (what GARCH does NOT explain)
# 4. Squared return (volatility proxy)
# 5. Absolute return (another volatility proxy)
# 6. Rolling 5-day and 21-day volatility (short and medium memory)

df = garch_df.copy()
df['Sq_Return']    = df['Log_Return'] ** 2
df['Abs_Return']   = df['Log_Return'].abs()
df['RollVol_5']    = df['Log_Return'].rolling(5).std()
df['RollVol_21']   = df['Log_Return'].rolling(21).std()
df = df.dropna()

print(f"\nFeature matrix shape: {df.shape}")
print(f"Features: {list(df.columns)}")

In [ ]:
# ── Target variable: next-day conditional volatility ──────────
# We shift Cond_Vol by -1 so the model learns to predict
# tomorrow's volatility from today's features
df['Target'] = df['Cond_Vol'].shift(-1)
df = df.dropna()

feature_cols = ['Log_Return', 'Cond_Vol', 'Std_Residual',
                'Sq_Return', 'Abs_Return', 'RollVol_5', 'RollVol_21']
X = df[feature_cols].values
y = df['Target'].values

print(f"\nX shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"Target range: {y.min():.6f} to {y.max():.6f}")

In [ ]:
# ── Train / test split (80% train, 20% test) ──────────────────
# Important: no shuffling — time series must stay sequential
split_idx = int(len(X) * 0.80)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

train_dates = df.index[:split_idx]
test_dates  = df.index[split_idx:]

print(f"\nTrain: {len(X_train):,} rows  ({train_dates[0].date()} to {train_dates[-1].date()})")
print(f"Test:  {len(X_test):,} rows  ({test_dates[0].date()} to {test_dates[-1].date()})")

In [ ]:
# ── Scale features ────────────────────────────────────────────
scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_train_sc = scaler_X.fit_transform(X_train)
X_test_sc  = scaler_X.transform(X_test)
y_train_sc = scaler_y.fit_transform(y_train.reshape(-1, 1)).flatten()
y_test_sc  = scaler_y.transform(y_test.reshape(-1, 1)).flatten()

# ── Reshape for LSTM: (samples, timesteps, features) ─────────
# We use a lookback window of 20 trading days (~1 month)
LOOKBACK = 20

def create_sequences(X, y, lookback):
    Xs, ys = [], []
    for i in range(lookback, len(X)):
        Xs.append(X[i-lookback:i])
        ys.append(y[i])
    return np.array(Xs), np.array(ys)

X_train_seq, y_train_seq = create_sequences(X_train_sc, y_train_sc, LOOKBACK)
X_test_seq,  y_test_seq  = create_sequences(X_test_sc,  y_test_sc,  LOOKBACK)

print(f"\nSequence shapes:")
print(f"X_train_seq: {X_train_seq.shape}  (samples, timesteps, features)")
print(f"X_test_seq:  {X_test_seq.shape}")

In [ ]:
# ── Build LSTM model ──────────────────────────────────────────
model = Sequential([
    LSTM(64, return_sequences=True,
         input_shape=(LOOKBACK, len(feature_cols))),
    Dropout(0.2),
    LSTM(32, return_sequences=False),
    Dropout(0.2),
    Dense(16, activation='relu'),
    Dense(1)
])

model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
              loss='mse')

model.summary()

# ── Callbacks ─────────────────────────────────────────────────
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=15,
    restore_best_weights=True,
    verbose=1
)
reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=7,
    min_lr=1e-6,
    verbose=1
)

# ── Train LSTM ────────────────────────────────────────────────
print("\nTraining LSTM — this will take 2-5 minutes...")
history = model.fit(
    X_train_seq, y_train_seq,
    epochs=100,
    batch_size=32,
    validation_split=0.15,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

print(f"\nTraining complete. Best epoch: {np.argmin(history.history['val_loss'])+1}")

In [ ]:
# ── Generate predictions ──────────────────────────────────────
y_pred_sc = model.predict(X_test_seq, verbose=0).flatten()

# Inverse scale back to original units
y_pred = scaler_y.inverse_transform(y_pred_sc.reshape(-1, 1)).flatten()
y_true = scaler_y.inverse_transform(y_test_seq.reshape(-1, 1)).flatten()

# ── Evaluation metrics ────────────────────────────────────────
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
mae  = mean_absolute_error(y_true, y_pred)
# Directional accuracy — did the model predict the right direction?
dir_acc = np.mean(np.sign(np.diff(y_true)) == np.sign(np.diff(y_pred)))

print("\n=== LSTM Model Performance (Test Set) ===")
print(f"RMSE:                 {rmse:.6f}")
print(f"MAE:                  {mae:.6f}")
print(f"Directional accuracy: {dir_acc:.2%}")

In [ ]:
# Annualised equivalents
print(f"\nAnnualised RMSE: {rmse*np.sqrt(252)*100:.4f}%")
print(f"Annualised MAE:  {mae*np.sqrt(252)*100:.4f}%")

# Save predictions
pred_df = pd.DataFrame({
    'Date':      test_dates[LOOKBACK:],
    'True_Vol':  y_true,
    'LSTM_Pred': y_pred
}).set_index('Date')
pred_df.to_csv('data/lstm_predictions.csv')
print("\nLSTM predictions saved to data/lstm_predictions.csv")

In [ ]:
# ── Plot 06: Training history ─────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['loss'],     label='Train loss', color='#2166ac')
axes[0].plot(history.history['val_loss'], label='Val loss',   color='#d6604d')
axes[0].set_title('LSTM Training History — Loss Curves',
                  fontweight='bold', fontsize=12)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE Loss')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Predicted vs actual on test set
axes[1].scatter(y_true, y_pred, alpha=0.3, s=5,
                color='#2166ac', label='Predictions')
min_v, max_v = min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())
axes[1].plot([min_v, max_v], [min_v, max_v], 'r--',
             linewidth=1.5, label='Perfect forecast')
axes[1].set_title('LSTM Predicted vs Actual Volatility',
                  fontweight='bold', fontsize=12)
axes[1].set_xlabel('Actual Conditional Volatility')
axes[1].set_ylabel('LSTM Predicted Volatility')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('outputs/06_lstm_training.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: outputs/06_lstm_training.png")

In [ ]:
# ── Plot 07: Predicted vs actual volatility over time ─────────
fig2, ax = plt.subplots(figsize=(14, 5))
plot_dates = pred_df.index

ax.plot(plot_dates, pred_df['True_Vol']*100*np.sqrt(252),
        color='#2166ac', linewidth=0.8, alpha=0.8, label='Actual (GARCH) vol')
ax.plot(plot_dates, pred_df['LSTM_Pred']*100*np.sqrt(252),
        color='#d6604d', linewidth=0.8, alpha=0.8, label='LSTM forecast')
ax.set_title('LSTM Volatility Forecast vs Actual — Test Period',
             fontweight='bold', fontsize=12)
ax.set_ylabel('Annualised Volatility (%)')
ax.set_xlabel('Date')
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('outputs/07_lstm_forecast.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: outputs/07_lstm_forecast.png")